# PRÁCTICA 2 PLN
# Desarrollo de una aplicación de Procesamiento del Lenguaje Natural

Alumnos:

Javier García Fernández

Miguel Ángel Véliz Ayala

# 4. Análisis de subjetividad de los comentarios

In [ ]:
import json
import os
import numpy as np
from tqdm import tqdm
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Función para cargar los archivos JSON (reutilizada del ejercicio anterior)
def load_json_files():
    data = {}
    json_files = [
        'comments_cricket.json',
        'comments_formula1.json',
        'comments_hockey.json',
        'comments_nba.json',
        'comments_nfl.json',
        'comments_soccer.json',
        'comments_sports.json'
    ]

    for file_name in json_files:
        subreddit = file_name.replace('comments_', '').replace('.json', '')
        try:
            with open(file_name, 'r', encoding='utf-8') as f:
                data[subreddit] = json.load(f)
            print(f"Cargado {file_name}: {len(data[subreddit])} hilos")
        except FileNotFoundError:
            print(f"Advertencia: No se encontró el archivo {file_name}")

    return data

In [ ]:
# Clase para análisis de sentimientos
class SentimentAnalyzer:
    def __init__(self, model_name):
        print(f"Cargando modelo de sentimientos: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)

        # Obtener las etiquetas del modelo
        if hasattr(self.model.config, 'id2label'):
            self.labels = self.model.config.id2label
        else:
            # Etiquetas por defecto para modelos de sentimiento
            self.labels = {0: "negative", 1: "neutral", 2: "positive"}

        print(f"Modelo cargado con etiquetas: {self.labels}")

    def analyze(self, text):
        # Truncar texto si es muy largo (prevenir errores con secuencias demasiado largas)
        max_length = self.tokenizer.model_max_length
        if max_length > 1024:
            max_length = 1024  # Limitar a 1024 para no consumir demasiada memoria

        # Preparar el texto para el modelo
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(self.device)

        # Obtener predicciones
        with torch.no_grad():
            outputs = self.model(**inputs)

        # Calcular probabilidades con softmax
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

        # Obtener la etiqueta con mayor probabilidad
        predicted_class_id = torch.argmax(probs, dim=-1).item()
        predicted_label = self.labels[predicted_class_id]

        # Crear un diccionario con todas las probabilidades
        result = {
            "label": predicted_label,
            "probabilities": {}
        }

        # Guardar todas las probabilidades
        for class_id, label in self.labels.items():
            result["probabilities"][label] = probs[0][class_id].item()

        return result

In [ ]:
# Clase para análisis de emociones (es prácticamente idéntica a la anterior)
class EmotionAnalyzer:
    def __init__(self, model_name):
        print(f"Cargando modelo de emociones: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)

        # Obtener las etiquetas del modelo
        if hasattr(self.model.config, 'id2label'):
            self.labels = self.model.config.id2label
        else:
            # Etiquetas por defecto para modelos de emoción (ejemplo)
            self.labels = {0: "anger", 1: "joy", 2: "fear", 3: "sadness", 4: "surprise"}

        print(f"Modelo cargado con etiquetas: {self.labels}")

    def analyze(self, text):
        # Truncar texto si es muy largo
        max_length = self.tokenizer.model_max_length
        if max_length > 1024:
            max_length = 1024

        # Preparar el texto para el modelo
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(self.device)

        # Obtener predicciones
        with torch.no_grad():
            outputs = self.model(**inputs)

        # Calcular probabilidades con softmax
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

        # Obtener la etiqueta con mayor probabilidad
        predicted_class_id = torch.argmax(probs, dim=-1).item()
        predicted_label = self.labels[predicted_class_id]

        # Crear un diccionario con todas las probabilidades
        result = {
            "label": predicted_label,
            "probabilities": {}
        }

        # Guardar todas las probabilidades
        for class_id, label in self.labels.items():
            result["probabilities"][label] = probs[0][class_id].item()

        return result

La decisión de crear clases en lugar de funciones sueltas se debe a varios motivos. Uno de ellos es la configuración de los modelos. Las clases encapsulan todos los elementos necesarios para el análisis en un solo objeto, lo que permite reutilizar el mismo modelo sin necesidad de recargarlo constantemente. También nos permite una mayor legibilidad, al estar todo concentrado en el mismo sitio.

Esta práctica de crear clases para modelos es muy usada en el ámbito del aprendizaje profundo por su gran cantidad de ventajas y sobre todo comodida a largo plazo, así como reutilización de código en otros venideros.

In [ ]:
# Función para procesar comentarios con los analizadores
def process_comments_sentiment_emotion(data, sentiment_analyzer=None, emotion_analyzer=None, batch_size=32):
    total_comments = 0
    processed_comments = 0

    # Contar el total de comentarios para la barra de progreso
    for subreddit, threads in data.items():
        for thread in threads:
            total_comments += len(thread['comments'])

    print(f"Procesando {total_comments} comentarios en total...")

    # Procesar comentarios por lotes para optimizar rendimiento
    for subreddit in tqdm(data.keys(), desc="Subreddits"):
        for thread_idx, thread in enumerate(tqdm(data[subreddit], desc=f"Hilos de {subreddit}", leave=False)):

            # Procesar comentarios en este hilo
            for comment_idx, comment in enumerate(thread['comments']):
                # Verificar si ya se han procesado estos comentarios
                if ('sentiment' in comment and sentiment_analyzer is None) or ('emotion' in comment and emotion_analyzer is None):
                    continue

                # Obtener el texto del comentario
                text = comment['comment']

                # Analizar sentimiento si se proporcionó un analizador
                if sentiment_analyzer is not None and 'sentiment' not in comment:
                    try:
                        sentiment_result = sentiment_analyzer.analyze(text)
                        data[subreddit][thread_idx]['comments'][comment_idx]['sentiment'] = sentiment_result
                    except Exception as e:
                        print(f"Error al analizar sentimiento: {e}")
                        data[subreddit][thread_idx]['comments'][comment_idx]['sentiment'] = {
                            "label": "error",
                            "probabilities": {}
                        }

                # Analizar emoción si se proporcionó un analizador
                if emotion_analyzer is not None and 'emotion' not in comment:
                    try:
                        emotion_result = emotion_analyzer.analyze(text)
                        data[subreddit][thread_idx]['comments'][comment_idx]['emotion'] = emotion_result
                    except Exception as e:
                        print(f"Error al analizar emoción: {e}")
                        data[subreddit][thread_idx]['comments'][comment_idx]['emotion'] = {
                            "label": "error",
                            "probabilities": {}
                        }

                processed_comments += 1

                # Mostrar progreso cada 100 comentarios
                if processed_comments % 100 == 0:
                    print(f"Procesados {processed_comments}/{total_comments} comentarios")

    print(f"Análisis completo: {processed_comments} comentarios procesados")
    return data

In [ ]:
# Función para guardar los resultados en archivos JSON
def save_json_data(data):
    for subreddit, threads in data.items():
        output_file = f"comments_{subreddit}_sentiment_emotion.json"
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(threads, f, ensure_ascii=False, indent=4)
        print(f"Datos guardados en {output_file}")

In [ ]:
# Función para analizar la distribución de sentimientos y emociones
def analyze_sentiment_emotion_distribution(data):
    sentiment_stats = {}
    emotion_stats = {}

    for subreddit, threads in data.items():
        sentiment_stats[subreddit] = {}
        emotion_stats[subreddit] = {}

        for thread in threads:
            for comment in thread['comments']:
                # Contar sentimientos
                if 'sentiment' in comment:
                    sentiment = comment['sentiment']['label']
                    sentiment_stats[subreddit][sentiment] = sentiment_stats[subreddit].get(sentiment, 0) + 1

                # Contar emociones
                if 'emotion' in comment:
                    emotion = comment['emotion']['label']
                    emotion_stats[subreddit][emotion] = emotion_stats[subreddit].get(emotion, 0) + 1

    # Mostrar estadísticas de sentimientos
    print("\n=== Distribución de Sentimientos por Subreddit ===")
    for subreddit, counts in sentiment_stats.items():
        print(f"\nSubreddit: r/{subreddit}")
        total = sum(counts.values())
        if total > 0:
            for sentiment, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
                print(f"  {sentiment}: {count} ({count/total*100:.1f}%)")
        else:
            print("  No se encontraron datos de sentimiento")

    # Mostrar estadísticas de emociones
    print("\n=== Distribución de Emociones por Subreddit ===")
    for subreddit, counts in emotion_stats.items():
        print(f"\nSubreddit: r/{subreddit}")
        total = sum(counts.values())
        if total > 0:
            for emotion, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
                print(f"  {emotion}: {count} ({count/total*100:.1f}%)")
        else:
            print("  No se encontraron datos de emoción")

    return sentiment_stats, emotion_stats

In [ ]:
# Función principal
def main():
    print("Análisis de subjetividad de comentarios de Reddit")
    print("=" * 70)

    # Paso 1: Cargar los datos
    data = load_json_files()

    if not data:
        print("No se encontraron archivos de datos. Abortando.")
        return

    # Paso 2: Configurar los analizadores
    # Usaremos el modelo en inglés para sentimientos (dado que los comentarios están en inglés)
    sentiment_model = "cardiffnlp/twitter-roberta-base-sentiment"
    sentiment_analyzer = SentimentAnalyzer(sentiment_model)

    # Usaremos el modelo en inglés para emociones
    emotion_model = "michellejieli/emotion_text_classifier"
    emotion_analyzer = EmotionAnalyzer(emotion_model)

    # Paso 3: Procesar los comentarios
    data = process_comments_sentiment_emotion(
        data,
        sentiment_analyzer=sentiment_analyzer,
        emotion_analyzer=emotion_analyzer
    )

    # Paso 4: Analizar la distribución de sentimientos y emociones
    sentiment_stats, emotion_stats = analyze_sentiment_emotion_distribution(data)

    # Paso 5: Guardar los resultados
    save_json_data(data)

if __name__ == "__main__":
    main()

Análisis de subjetividad de comentarios de Reddit
Cargado comments_cricket.json: 180 hilos
Cargado comments_formula1.json: 180 hilos
Cargado comments_hockey.json: 180 hilos
Cargado comments_nba.json: 180 hilos
Cargado comments_nfl.json: 180 hilos
Cargado comments_soccer.json: 160 hilos
Cargado comments_sports.json: 104 hilos
Cargando modelo de sentimientos: cardiffnlp/twitter-roberta-base-sentiment


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

Procesados 7300/7424 comentarios



Hilos de sports:  97%|█████████▋| 101/104 [04:38<00:07,  2.59s/it]

Procesados 7400/7424 comentarios



Subreddits: 100%|██████████| 7/7 [46:29<00:00, 398.51s/it]


Análisis completo: 7424 comentarios procesados

=== Distribución de Sentimientos por Subreddit ===

Subreddit: r/cricket
  LABEL_1: 472 (42.7%)
  LABEL_0: 368 (33.3%)
  LABEL_2: 258 (23.3%)
  error: 7 (0.6%)

Subreddit: r/formula1
  LABEL_1: 499 (44.6%)
  LABEL_0: 399 (35.7%)
  LABEL_2: 217 (19.4%)
  error: 3 (0.3%)

Subreddit: r/hockey
  LABEL_0: 468 (42.0%)
  LABEL_1: 387 (34.8%)
  LABEL_2: 256 (23.0%)
  error: 2 (0.2%)

Subreddit: r/nba
  LABEL_1: 449 (40.0%)
  LABEL_0: 391 (34.8%)
  LABEL_2: 282 (25.1%)
  error: 1 (0.1%)

Subreddit: r/nfl
  LABEL_0: 454 (38.8%)
  LABEL_1: 403 (34.4%)
  LABEL_2: 308 (26.3%)
  error: 5 (0.4%)

Subreddit: r/soccer
  LABEL_1: 420 (41.6%)
  LABEL_0: 416 (41.2%)
  LABEL_2: 171 (16.9%)
  error: 2 (0.2%)

Subreddit: r/sports
  LABEL_0: 401 (51.0%)
  LABEL_1: 249 (31.7%)
  LABEL_2: 135 (17.2%)
  error: 1 (0.1%)

=== Distribución de Emociones por Subreddit ===

Subreddit: r/cricket
  neutral: 778 (70.4%)
  joy: 88 (8.0%)
  disgust: 71 (6.4%)
  sadness: 55 (5

A partir de la salida proporcionada, podemos extraer una serie de observaciones relevantes que nos van a permitir describir el proceso de análisis de subjetividad en los comentarios de Reddit. En esta ocasión, como en anteriores, se han cargado aproximadamente 180 hilos por cada subreddit, alcanzando un total de 7424 comentarios procesados, lo que representa una muestra amplia y representativa del contenido publicado en estos foros.

Para este análisis de subjetividad, se han empleado dos modelos distintos, cada uno con un propósito concreto:

- Modelo de sentimientos
  - Hemos usado el modelo *cardiffnlp/twitter-roberta-base-sentiment*. Este modelo está basado en RoBERTa, entrenado sobre datos de Twitter. Proporciona una clasificación en tres categorías de sentimientos.
    - LABEL_0 ➡ Negativo
    - LABEL_1 ➡ Neutro
    - LABEL_2 ➡ Positivo

  Este modelo entrenado en tweets puede ser muy pertinente para nuestro caso de estudio pues la similitud entre reddits y tweets es significativa.

  En cuanto a los resultados obtenidos por subreddit, estos son los siguientes:

    
  | Subreddit      | Negativo (%) | Neutro (%)   | Positivo (%) | Sentimiento predominante |
  |----------------|--------------|--------------|---------------|---------------------------|
  | **r/cricket**  | 33.3%        | **42.7%**    | 23.3%         | Neutro                   |
  | **r/formula1** | 35.7%        | **44.6%**    | 19.4%         | Neutro                   |
  | **r/hockey**   | **42.0%**    | 34.8%        | 23.0%         | Negativo                 |
  | **r/nba**      | 34.8%        | **40.0%**    | 25.1%         | Neutro                   |
  | **r/nfl**      | **38.8%**    | 34.4%        | 26.3%         | Negativo                 |
  | **r/soccer**   | 41.2%        | **41.6%**    | 16.9%         | Neutro                   |
  | **r/sports**   | **51.0%**    | 31.7%        | 17.2%         | Negativo                 |


Como se puede observar, la mayoría de subreddits presentan una distribución con predominancia del sentimiento neutro, salvo en r/hockey, r/nfl y r/sports, donde se impone el sentimiento negativo de forma clara. Esto podría estar relacionado con la naturaleza de los comentarios, que en estos subreddits tienden a ser más críticos o polémicos. Además, tratándose de deportes, el cual es un tema que tiende a ser muy fulgurante entre sus seguidores, pueden tener sentido dichos resultados, ya que las personas suelen defender su posición y/o equipo con bastante determinación, en cuanto a lo que deporte se refiere. Por otro lado, el sentimiento positivo, en general, es el menos frecuente, oscilando entre el 16.9% y el 26.3% según el subreddit.


---

- Modelo de emociones
  - Se ha empleado el modelo *michellejieli/emotion_text_classifier*, que clasifica en siete emociones.
    - Anger
    - Disgust
    - Fear
    - Joy
    - Neutral
    - Sadness
    - Surprise

  En cuanto a los resultados obtenidos por subreddit, estos son los siguientes:

| Subreddit      | Neutral (%) | Joy (%) | Disgust (%) | Anger (%) | Sadness (%) | Surprise (%) | Fear (%) | Emoción predominante |
| -------------- | ----------- | ------- | ----------- | --------- | ----------- | ------------ | -------- | -------------------- |
| **r/cricket**  | **70.4%**   | 8.0%    | 6.4%        | 4.9%      | 5.0%        | 3.8%         | 1.5%     | Neutral              |
| **r/formula1** | **74.5%**   | 5.5%    | 5.7%        | 6.5%      | 3.5%        | 3.2%         | 1.0%     | Neutral              |
| **r/hockey**   | **64.1%**   | 9.5%    | 8.2%        | 6.6%      | 4.6%        | 4.9%         | 2.2%     | Neutral              |
| **r/nba**      | **70.3%**   | 9.4%    | 7.7%        | 4.5%      | 3.6%        | 3.7%         | 0.8%     | Neutral              |
| **r/nfl**      | **65.5%**   | 9.8%    | 8.8%        | 4.5%      | 4.8%        | 5.7%         | 0.9%     | Neutral              |
| **r/soccer**   | **61.0%**   | 10.5%   | 10.4%       | 8.2%      | 5.9%        | 2.8%         | 1.2%     | Neutral              |
| **r/sports**   | **58.7%**   | 8.4%    | 14.5%       | 7.0%      | 4.5%        | 5.7%         | 1.3%     | Neutral              |

En todos los subreddits, la emoción más común es neutral, superando ampliamente el 50% de los comentarios en todos los casos. Esto es coherente con el tipo de contenido que suele compartirse en Reddit: muchos comentarios informan, opinan sin carga emocional o simplemente discuten datos objetivos.

Sin embargo, existen diferencias significativas en las emociones secundarias:

r/sports presenta una proporción considerable de disgust (14.5%), lo que podría señalar descontento o desacuerdo entre usuarios.

r/soccer y r/nfl muestran un nivel destacable de joy y disgust, lo que sugiere una polarización emocional en los comentarios.

r/formula1 muestra el mayor grado de neutralidad emocional (74.5%).

---

Conclusión general

El análisis combinado de sentimientos y emociones revela que, si bien la mayoría de los comentarios son neutros en tono y contenido, los subreddits difieren en el tipo y la intensidad de emociones expresadas. En particular, r/sports, r/soccer y r/hockey muestran más diversidad emocional y mayor carga subjetiva, mientras que r/formula1 y r/cricket se mantienen más contenidos en su tono.

Esta información puede resultar útil para entender mejor las dinámicas comunicativas de cada comunidad, así como para identificar espacios más polarizados o con mayor carga emocional.